# Layout-aware extraction

**Module 2 — Lesson 4**

In Module 1, you called `export_to_text()` to pull text from PDFs -- but alongside that text, Docling was simultaneously building a richer document model. `doc.texts` exposes the structured part of that model: an ordered list of discrete layout items, one per visual element on the page.

For email headers, that means each label and each value can arrive as its own separate block -- `'From:'` is one item, `'Matthew Lenhart'` is the next -- rather than a single string you have to split apart.

Whether Docling achieves that segmentation depends on the page layout. This notebook explores where it works, where it doesn't, and what that means for your parsing strategy.

## 1. Setup

In [1]:
%pip install docling pandas -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import time
import pandas as pd
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, LayoutOptions, DOCLING_LAYOUT_EGRET_XLARGE
from docling.document_converter import DocumentConverter, PdfFormatOption

PDF_DIR  = Path("enron_pdfs")
TEXT_DIR = Path("data/extracted_text")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
txt_files = sorted(TEXT_DIR.glob("*.txt"))

# Reference PDFs
CLEAN_PDF    = PDF_DIR / "E0048ADF3.pdf"   # clean digital text layer
SCANNED_PDF  = PDF_DIR / "E61D04918.pdf"   # scanned, moderate OCR errors
SCAN_GOOD    = PDF_DIR / "E00CF8AE9.pdf"   # scanned, good OCR (but different layout)
IMAGE_PDF    = PDF_DIR / "E0033CF3B.pdf"    # no text layer at all

print(f"PDFs:  {len(pdf_files)}")
print(f".txt:  {len(txt_files)}")

## 2. Converter configuration

The configuration is similar to Module 1 -- OCR on, same format options structure -- but here we explicitly set the layout model to `EGRET_XLARGE` for finer-grained segmentation. The default layout model works, but the larger model produces more discrete blocks, which matters when we want each header label and value as its own item.

The difference in this notebook comes after the conversion: instead of calling `export_to_text()`, you reach into `doc.texts`.

In [3]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr             = True
pipeline_options.generate_picture_images = False
pipeline_options.layout_options = LayoutOptions(
    model_spec=DOCLING_LAYOUT_EGRET_XLARGE
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


Pre-load the pipeline models now so the first `convert()` call doesn't carry the model-loading cost. This is the slow step — **expect 30–60 seconds** while Docling initialises OCR and layout detection. Everything after this runs at ~1 email/sec.

In [4]:
converter.initialize_pipeline(InputFormat.PDF)
print("Models loaded.")

Models loaded.


## 3. What doc.texts Contains

With models loaded and the converter ready, run it on the clean reference PDF and inspect `doc.texts` directly — first count the blocks to see the overall shape, then print the first ten to see exactly what Docling placed there and in what order.

In [5]:
result = converter.convert(str(CLEAN_PDF))
doc = result.document

print(f"File:           {CLEAN_PDF.name}")
print(f"Text blocks:    {len(doc.texts)}")

File:           E0048ADF3.pdf
Text blocks:    21


In [6]:
# Inspect the text blocks (non-table content)
for i, item in enumerate(doc.texts[:20]):
    print(f"[{i}] {item.text!r}")

[0] 'CONFIDENTIAL'
[1] 'Enron Corp.'
[2] 'Case No. EC-2002-01038'
[3] 'Doc No. E0048ADF3'
[4] 'Date: 01/15/2003'
[5] 'ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.'
[6] 'SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.'
[7] 'RELEASE IN FULL'
[8] 'From:'
[9] 'Nemec, Gerald <gerald.nemec@enron.com>'
[10] 'Sent:'
[11] 'Wed, 17 Oct 2001 13:11:22 -0700 (PDT)'
[12] 'To:'
[13] 'Lucci, Paul T. <t..lucci@enron.com>'
[14] 'Cc:'
[15] 'Tycholiz, Barry <barry.tycholiz@enron.com>, Williams, Jason R'
[16] '(Credit) <.williams@enron.com>'
[17] 'Subject:'
[18] 'Kennedy Amendment'
[19] 'Here is the latest.'


Docling segments the page into discrete blocks -- boilerplate lines, header labels, header values, and body text each get their own item. In this run, the first several blocks are boilerplate, followed by the email header: each label (`From:`, `Sent:`, `To:`, `Cc:`, `Subject:`) and each value as separate items.

Note that long values can wrap across multiple blocks -- the Cc value `Tycholiz, Barry <barry.tycholiz@enron.com>, Williams, Jason R` is split from its continuation `(Credit) <.williams@enron.com>` across two blocks.

This structured segmentation is what makes layout-aware extraction different from flat text. Instead of parsing one continuous string, you can walk the blocks in order and match labels to values.

## 4. Extracting email fields

Walk `doc.texts` in reading order. When a block matches a label pattern (`From:`, `Sent:`, etc.), start collecting. Everything after that label — whether inline or in subsequent blocks — is that field's value, until the next label appears.

Multi-line values (like long recipient lists) may span several blocks. The extractor collects all of them.


In [7]:
FIELD_RE = re.compile(r'^(From|Sent|To|Cc|Subject)\s*:', re.IGNORECASE)

FIELD_NAMES = {'from': 'from', 'sent': 'date', 'to': 'to', 'cc': 'cc', 'subject': 'subject'}


def extract_fields(doc):
    """Extract email header fields from a Docling document object."""
    result = {}
    items = [item.text.strip() for item in doc.texts]
    current = None

    for text in items:
        m = FIELD_RE.match(text)
        if m:
            label = m.group(1).lower()
            current = FIELD_NAMES[label]
            after = text[m.end():].strip()
            result[current] = after
        elif current:
            result[current] += ' ' + text

        if current == 'subject' and result.get('subject'):
            break

    return result


# Test on the clean PDF
fields = extract_fields(doc)
print()
for field, value in fields.items():
    print(f"  {field:<10} {value!r}")



  from       ' Nemec, Gerald <gerald.nemec@enron.com>'
  date       ' Wed, 17 Oct 2001 13:11:22 -0700 (PDT)'
  to         ' Lucci, Paul T. <t..lucci@enron.com>'
  cc         ' Tycholiz, Barry <barry.tycholiz@enron.com>, Williams, Jason R (Credit) <.williams@enron.com>'
  subject    ' Kennedy Amendment'


All five fields extracted from the clean PDF. The Cc value correctly collects the continuation block — `Tycholiz, Barry` and `Williams, Jason R (Credit)` are joined into one value.

The leading spaces in the values come from Docling's text extraction. A real pipeline would strip these.


## 5. Scanned PDFs: where segmentation varies

Clean digital PDFs produce reliable segmentation — every label and value gets its own block. Scanned PDFs are less predictable. Docling's layout model runs on the page image, and the quality of segmentation depends on the visual structure of each scan.

Some scanned files segment well. Others merge values across fields, or split names across blocks. The next cell runs the same extractor on two scanned files to see what happens.


In [8]:
# Scanned file with moderate OCR errors
result_scanned = converter.convert(str(SCANNED_PDF))
doc_scanned = result_scanned.document

print(f"=== {SCANNED_PDF.name} (moderate OCR) ===")
print(f"  Text blocks: {len(doc_scanned.texts)}")
for i, item in enumerate(doc_scanned.texts[:20]):
    print(f"  [{i}] {item.text[:70]!r}")

print()

# Scanned file with clean OCR but multi-line recipients
result_good = converter.convert(str(SCAN_GOOD))
doc_good = result_good.document

print(f"=== {SCAN_GOOD.name} (good OCR) ===")
print(f"  Text blocks: {len(doc_good.texts)}")
for i, item in enumerate(doc_good.texts[:20]):
    print(f"  [{i}] {item.text[:70]!r}")

=== E61D04918.pdf (moderate OCR) ===
  Text blocks: 54
  [0] 'CONFIDENTIAL'
  [1] 'Enron Corp.'
  [2] 'Case No. EC-26002-016038'
  [3] 'Doc No. Es6lbo4918'
  [4] 'Bate: O1/15/2003'
  [5] 'ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.'
  [6] 'SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.'
  [7] 'RELEASE IN PART'
  [8] 'From:'
  [9] 'Ker i Thompson <ker i.thompson@enron,.'
  [10] 'com>'
  [11] 'Sent:'
  [12] 'Wed, 22 Nov 2000 05:41:00 -0800 (PST)'
  [13] 'To:'
  [14] 'Kate Symes <kate.symes@enron.com>'
  [15] 'Subject :'
  [16] 'natsource checkout'
  [17] '465810'
  [18] 'mat strike price 70.00'
  [19] '8.00 premium'

=== E00CF8AE9.pdf (good OCR) ===
  Text blocks: 57
  [0] 'CONFIDENTIAL'
  [1] 'Enron Corp.'
  [2] 'Case No.  EC-2002-01038'
  [3] 'Doc No.  EOOCF8AE9'
  [4] 'Date:  01/15/2003'
  [5] 'ENRON CORP.  -  PRODUCED PURSUANT TO FERC SUBPOENA.'
  [6] 'SUBJECT TO PROTECTIVE ORDER.  CONFIDENTIAL TREATMENT REQUESTED.'
  [7] 'RELEASE IN PART'
  [8] 'From:'
  [9] 'A

In [9]:
# Try the field extractor on both
fields_scanned = extract_fields(doc_scanned)
fields_good = extract_fields(doc_good)

print(f"=== {SCANNED_PDF.name}: {len(fields_scanned)} fields ===")
for k, v in fields_scanned.items():
    print(f"  {k:<10} {v[:60]!r}")

print(f"\n=== {SCAN_GOOD.name}: {len(fields_good)} fields ===")
if fields_good:
    for k, v in fields_good.items():
        print(f"  {k:<10} {v[:60]!r}")
else:
    print("  (no fields — headers collapsed into boilerplate blocks)")

=== E61D04918.pdf: 4 fields ===
  from       ' Ker i Thompson <ker i.thompson@enron,. com>'
  date       ' Wed, 22 Nov 2000 05:41:00 -0800 (PST)'
  to         ' Kate Symes <kate.symes@enron.com>'
  subject    ' natsource checkout'

=== E00CF8AE9.pdf: 4 fields ===
  from       ' Amr Ibrahim <amr.ibrahim@enron.com>'
  date       ' Thu,  5  Jul  2001  01:02:00  -0700  (PDT)'
  to         ' Richard Shapiro <richard.shapiro@enron.com>,  Lisa Yoho <li'
  subject    ' Fireball Coal Project -  RAC Meeting June 29'


`E61D04918.pdf` (moderate OCR) segments into many blocks -- each label and value separated, though OCR has garbled some content (`Ker i Thompson` for `Kerri Thompson`, `com>` on a separate block from the email address). The extractor finds four of five fields.

`E00CF8AE9.pdf` (good OCR) also segments well. The To field picks up continuation blocks correctly -- `Richard Shapiro` and `Lisa Yoho` are joined into one value, along with the `[REDACTED]` marker.

The exact block counts and segmentation depend on the Docling version and OCR backend. What matters is the pattern: labels and values as separate blocks, with variation across scans.

## 6. Measuring Field Coverage

The extractor works on one email. 

Before committing to a full corpus run, however, you may want to measure coverage. 

The cell below runs a coverage test on a 100-PDF sample to see what proportion of fields survive and whether the approach is worth the processing time.

> **This runs the full Docling pipeline on 100 PDFs — expect ~100 seconds.**

In [10]:
sample = pdf_files[:100]
rows = []

for pdf in sample:
    result = converter.convert(str(pdf))
    doc    = result.document
    fields = extract_fields(doc)
    fields["file"] = pdf.name
    rows.append(fields)

df_results = pd.DataFrame(rows)

field_cols = ["from", "date", "to", "cc", "subject"]
present = {c: df_results[c].notna() & (df_results[c] != "") for c in field_cols if c in df_results}

print(f"Field coverage across {len(sample)} PDFs:")
for field, mask in present.items():
    print(f"  {field:<10} {mask.sum()}/{len(sample)} ({mask.mean()*100:.0f}%)")

Field coverage across 100 PDFs:
  from       75/100 (75%)
  date       72/100 (72%)
  to         68/100 (68%)
  cc         15/100 (15%)
  subject    85/100 (85%)


Check your coverage numbers. In our run, layout-aware extraction recovered 70-85% of fields across the 100-file sample. Your numbers may differ depending on the Docling version and OCR backend.

The remaining fields are lost to segmentation inconsistencies -- merged blocks, OCR splitting names, or labels not being recognized. This is a useful baseline, but not reliable enough for graph import on its own. In the next lessons, regex templates on the pre-extracted text achieve 98%+ coverage.

In a real-world workflow, you might use both -- Docling for its layout awareness, and regex to further parse the segments it identifies.

In [11]:
cols = ["file"] + [c for c in field_cols if c in df_results]
print(df_results[cols].to_string(index=False))

         file                                                                                                         from                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                date                                                                                                                                                                                                                                                                                                                                                          

## 7. Speed: Docling vs Regex on Pre-Extracted Text

Docling re-processes the PDFs from scratch — the `.txt` files from Module 1 cannot be reused. Here's what that cost looks like against a simple regex on the pre-extracted text.

In [12]:
N = 50
sample_pdfs = pdf_files[:N]
sample_txts = txt_files[:N]

# --- Docling structured extraction (from PDFs) ---
start = time.perf_counter()
for p in sample_pdfs:
    result = converter.convert(str(p))
    _      = extract_fields(result.document)
elapsed_docling = time.perf_counter() - start

# --- Regex on pre-extracted .txt files ---
HEADER_RE = re.compile(
    r"^(From|Sent|To|Cc|Subject):\s*(.+)$",
    re.MULTILINE | re.IGNORECASE
)

def parse_txt(text):
    return {
        m.group(1).lower(): m.group(2).strip()
        for m in HEADER_RE.finditer(text)
    }

start = time.perf_counter()
for p in sample_txts:
    text = p.read_text(errors="replace")
    _    = parse_txt(text)
elapsed_regex = time.perf_counter() - start

rate_docling = N / elapsed_docling
rate_regex   = N / elapsed_regex

print(f"Sample: {N} emails")
print()
print(f"{'Method':<40} {'Time':>8}   {'emails/sec':>11}")
print("-" * 63)
print(f"{'Docling (structured from PDF)':<40} {elapsed_docling:>7.1f}s   {rate_docling:>10.1f}")
print(f"{'Regex on pre-extracted .txt':<40} {elapsed_regex:>7.3f}s   {rate_regex:>10.0f}")
print()
print(f"Regex is ~{rate_regex / rate_docling:.0f}x faster.")
print()
print(f"Projected time for all 5,000 emails:")
print(f"  Docling:  {5000 / rate_docling / 60:.0f} min")
print(f"  Regex:    {5000 / rate_regex:.1f} sec")

Sample: 50 emails

Method                                       Time    emails/sec
---------------------------------------------------------------
Docling (structured from PDF)               34.1s          1.5
Regex on pre-extracted .txt                0.012s         4032

Regex is ~2751x faster.

Projected time for all 5,000 emails:
  Docling:  57 min
  Regex:    1.2 sec


## 8. When to use Docling for email parsing

The speed test puts the tradeoff in concrete numbers. For our email corpus — single-column, already extracted to text files — regex on the `.txt` files is thousands of times faster and more reliable.

Layout-aware extraction is most valuable when you're starting from raw PDFs with complex layouts (multi-column, tables, forms) where flat text extraction loses structure. For simple emails, it's not the right tool.


## 9. Docling's structured extraction API

Docling also offers a `DocumentExtractor` that takes a completely different approach. Instead of walking layout blocks, it renders each page as an image and sends it to a local vision-language model (NuExtract-2.0-2B) with a schema template. The model reads the page image and fills in the fields.

This is schema-driven extraction -- you define a Pydantic model describing what you want, and the VLM figures out how to get it from the page. No block walking, no regex, no layout assumptions.

> **Note:** This API is currently beta. The first run downloads a ~1.4GB model. The cell below installs `qwen-vl-utils` (required by the VLM) and removes the `av` package it pulls in to avoid a library conflict with OpenCV.

In [13]:
# qwen-vl-utils is required by NuExtract. It pulls in the av package,
# which conflicts with OpenCV's bundled libavdevice on some platforms.
# Uninstalling av after install works around this. If you don't see the
# conflict (no objc warnings), you can skip the uninstall line.
%pip install qwen-vl-utils -q
%pip uninstall av -y -q

import warnings
import os

warnings.filterwarnings("ignore", message="The extract API")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

from pydantic import BaseModel, Field
from typing import Optional, List
from docling.document_extractor import DocumentExtractor

# Define the schema -- the VLM will fill these fields from the page image
class EmailFields(BaseModel):
    sender: Optional[str] = Field(None, description="The From: field - who sent the email")
    date: Optional[str] = Field(None, description="The Sent: field - when the email was sent")
    recipients: Optional[List[str]] = Field(None, description="The To: field - who received the email")
    cc: Optional[List[str]] = Field(None, description="The Cc: field - who was copied")
    subject: Optional[str] = Field(None, description="The Subject: field - the email subject line")

extractor = DocumentExtractor(allowed_formats=[InputFormat.PDF])

# Test on all three reference files
for name, label in [("E0048ADF3.pdf", "Clean digital"),
                     ("E61D04918.pdf", "Moderate OCR"),
                     ("E00CF8AE9.pdf", "Good scan")]:
    start = time.perf_counter()
    result = extractor.extract(source=str(PDF_DIR / name), template=EmailFields)
    elapsed = time.perf_counter() - start

    print(f"=== {label} ({name}) — {elapsed:.1f}s ===")
    for page_data in result.pages:
        print(f"  Page {page_data.page_no}: {page_data.extracted_data}")
    print()

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Clean digital (E0048ADF3.pdf) — 18.5s ===
  Page 1: {'sender': 'Nemec, Gerald', 'date': '2001-10-17T13:11:22-07:00', 'recipients': ['Lucci, Paul T.', 'Tycholiz, Barry', 'Williams, Jason R (Credit)'], 'cc': ['Tycholiz, Barry', 'Williams, Jason R (Credit)'], 'subject': 'Kennedy Amendment'}

=== Moderate OCR (E61D04918.pdf) — 13.8s ===
  Page 1: {'sender': 'Kerri Thompson', 'date': '2000-11-22T05:41:00-08:00', 'recipients': ['Kate Symes'], 'cc': [], 'subject': 'natsource checkout'}

=== Good scan (E00CF8AE9.pdf) — 29.7s ===
  Page 1: {'sender': 'Amr Ibrahim', 'date': '2001-07-05T07:50:00', 'recipients': ['Richard Shapiro', 'Lisa Yoho'], 'cc': ['James D Steffes', 'Paul Dawson', "Nicholas O'Day", 'Steve Montovano', 'Chauncey Hood'], 'subject': 'RAC Meeting June 29'}
  Page 2: {'sender': 'AI', 'date': '2003-01-15', 'recipients': ['Enron Corp.', 'ENCom Europe', 'EES'], 'cc': ['Mike Beyer', 'S. Yaijima', 'Patrick Bastien', 'David Marks', 'Steve Monotvano'], 'subject': 'Enron Corp. - Produc

The `DocumentExtractor` uses a VLM to read the page image directly — no block walking, no segmentation dependencies. It fills a schema from what it sees on the page.

This approach bypasses the segmentation issues entirely. The trade-off is speed (~15-30s per PDF) and non-determinism — the VLM may hallucinate or misread boilerplate as content. Check the multi-page results: the model sometimes extracts the boilerplate header block on page 2 as if it were a separate email.

For documents with complex or unpredictable layouts — invoices, forms, handwritten annotations — this is worth the cost. For our email corpus, regex on pre-extracted text is faster and more reliable.

## Summary

- **`doc.texts` block walking** exposes Docling's layout model -- each visual region becomes a discrete block. On clean digital PDFs, labels and values segment reliably. On scanned PDFs, segmentation varies.
- **Multi-line values** (long recipient lists, wrapped emails) may span multiple blocks. The extractor needs to collect continuation blocks until the next label.
- **Field coverage** in our sample was ~70-85% -- sufficient for exploration, but not reliable enough for graph import. Your results may vary with Docling version and OCR backend.
- **`DocumentExtractor`** (VLM-based) bypasses segmentation entirely by reading page images. Slower but more reliable on difficult layouts.
- **Speed**: Docling ~1-2 emails/sec from PDF, regex ~4,000/sec on pre-extracted text. For simple emails, regex wins.
- For complex documents (invoices, forms, irregular layouts), layout-aware extraction is worth the speed cost.